In [11]:
import os
import dotenv

dotenv.load_dotenv()

if "TEXT_BIASED_DATA_PATH" not in os.environ:
    
    raise Exception("Please set the TEXT_BIASED_DATA_PATH environment variable to where you want to save your dataset")

DATA_DIR= os.environ["TEXT_BIASED_DATA_PATH"]
assert DATA_DIR, "Please set the TEXT_BIASED_DATA_PATH environment variable to your data directory"

# OUTPUT_DIR = os.path.join(DATA_DIR, "images/vsr") 
IMAGES_DIR = os.path.join(DATA_DIR, "images/vsr")
SAVE_DIR = os.path.join(DATA_DIR, "vsr")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)


In [12]:
from datasets import load_dataset

vsr_dataset = load_dataset("cambridgeltl/vsr_zeroshot")

In [13]:
vsr_dataset['train'][0]

{'image': '000000558388.jpg',
 'image_link': 'http://images.cocodataset.org/train2017/000000558388.jpg',
 'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'annotator_id': 35,
 'vote_true_validator_id': '[2, 67, 20]',
 'vote_false_validator_id': '[]'}

In [14]:
import os
import requests

In [15]:
# download image and return local path
def download_and_replace_image(example):
    image_url = example["image_link"]
    image_name = example["image"]
    local_path = os.path.join(IMAGES_DIR, image_name)
    
    # Download only if not already present
    if not os.path.exists(local_path):
        try:
            response = requests.get(image_url, timeout=10)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(response.content)
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")
            local_path = None  # or set to empty string if needed

    # Replace the 'image' field with the local file path
    return {
        "image_path": local_path,
        "caption": example["caption"],
        "label": example["label"],
        "relation": example["relation"],
        "subj": example["subj"],
        "obj": example["obj"]
    }

# Bias Through Undersampling
See create_biases.ipynb for the code that generates the data used in this notebook.

In [16]:
df_train = vsr_dataset["train"].to_pandas()

In [18]:
import pandas as pd

def undersample_by_relation(df,
                            keep_ratio=0.8,
                            seed=42):
    """
    For the listed relations:
      • keep **all** rows where label == minority_label
      • keep only `keep_ratio` of rows where label != minority_label
    All other relations are left untouched.
    """
    
    # Empty DataFrame to store the new data
    df_biased = pd.DataFrame(columns=df.columns)

    for rel, group in df.groupby('relation'):
        # Split into minority / majority within this relation
        group_count = group['label'].value_counts()
        
        minority_label = group_count.idxmin()
        majority_label = group_count.idxmax()
        
        group_min = group[group['label'] == minority_label]
        group_maj = group[group['label'] == majority_label]
        
        # Remove random from minority
        n_keep = int(len(group_min) * keep_ratio)
        n_drop = len(group_min) - n_keep
        if n_drop > 0:
            group_min_to_keep = group_min.sample(n=n_keep, random_state=seed)
        else:
            group_min_to_keep = group_min.sample(n=0, random_state=seed)
            
        new_group = pd.concat([group_maj, group_min_to_keep])
        df_biased = pd.concat([df_biased, new_group])  
        
    df_biased['label'] = df_biased['label'].astype('int')    
    
    return df_biased.reset_index(drop=True)


In [19]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd

# Load original dataset
dataset = load_dataset("cambridgeltl/vsr_zeroshot")

# Convert training set to pandas
df_train = pd.DataFrame(dataset['train'])

# Apply undersampling
df_train_biased = undersample_by_relation(df_train, keep_ratio=0.8)

# Rebuild DatasetDict with modified train set
biased_dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train_biased),
    "validation": dataset["validation"],
    "test": dataset["test"] 
})


In [20]:
# Map the dataset with image downloading
biased_dataset = biased_dataset.map(download_and_replace_image, num_proc=32)

Map (num_proc=32):   0%|          | 0/3137 [00:00<?, ? examples/s]

Map (num_proc=32):   0%|          | 0/340 [00:00<?, ? examples/s]

Map (num_proc=32):   0%|          | 0/1222 [00:00<?, ? examples/s]

In [21]:
# Remove unused columns (optional, in case you want a clean dataset)
biased_dataset = biased_dataset.remove_columns([col for col in biased_dataset.column_names['train'] if col not in ["image_path", "caption", "label", "relation", "subj", "obj"]])

In [22]:
# Sample
biased_dataset['train'][0]

{'caption': 'The pizza is above the dining table.',
 'label': 1,
 'relation': 'above',
 'subj': 'pizza',
 'obj': 'dining table',
 'image_path': '/scratch/izar/delsad/vlm_r1_text_bias/data/images/vsr/000000477195.jpg'}

In [23]:
# Save the dataset to disk
biased_dataset.save_to_disk(SAVE_DIR) # # change this to your path where you want to save the dataset

Saving the dataset (0/1 shards):   0%|          | 0/3137 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/340 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1222 [00:00<?, ? examples/s]